# 03 · Tensors and Autograd

**Part I — Foundations** · Estimated time: ~120 min

Notebook 02 · Neural Networks from Scratch (NumPy) ended with a diagnosis: writing the backward pass by hand is
instructive but does not scale. This notebook supplies the cure. We first learn to use PyTorch's
**tensor** — a NumPy array that can live on a GPU and remember how it was computed — and then we build the
missing piece ourselves: a small **automatic differentiation** engine in the spirit of Andrej Karpathy's
micrograd, complete with a computational graph, a topological sort and a `backward()` that produces every
gradient for free. Having built one, we study `torch.autograd`, the industrial version, in enough depth
to use it correctly and to debug it when it complains. The notebook closes by rewriting the MNIST
classifier of Notebook 02 in a few dozen lines of tensors plus autograd, and by previewing `nn.Module`
and `torch.optim`, the abstractions that the rest of the course is built on.

## Learning objectives

- Create, inspect, index, reshape and combine tensors, and explain the difference between a view and a copy.
- Apply the broadcasting rules to predict the shape of any element-wise operation, and use `dim`/`keepdim` reductions and `einsum` confidently.
- Explain forward- versus reverse-mode automatic differentiation and why reverse mode is the right tool for training neural networks.
- Implement a scalar autograd engine (`Value`) with a topological-sort `backward()`, verify it against finite differences, and train a tiny MLP with it.
- Use `requires_grad`, `backward`, `.grad`, `torch.no_grad`, `detach` and `torch.autograd.grad` correctly, and diagnose in-place-modification errors.
- Compute vector-Jacobian products, higher-order derivatives, Jacobians and Hessians with `torch.autograd` and `torch.func`.
- Write a custom `torch.autograd.Function` with a hand-written backward and check it with `gradcheck`.
- Train linear regression, logistic regression and an MNIST MLP with raw tensors and autograd, and read the resulting code as a preview of `nn.Module` and `torch.optim`.

## Prerequisites

- Notebook 01 · Mathematical Foundations (Jacobians and the chain rule, broadcasting, numerical differentiation)
- Notebook 02 · Neural Networks from Scratch (NumPy) (the forward and backward pass of an MLP, softmax cross-entropy, the training loop)

## Contents

1. Tensors
2. Computational graphs and reverse-mode automatic differentiation
3. A scalar autograd engine
4. `torch.autograd`
5. Training with raw tensors and autograd
6. Preview: `nn.Module` and `torch.optim`
- Exercises · Summary · Further reading

In [ ]:
import math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
torch.set_printoptions(precision=4, sci_mode=False)
print(f"FAST_RUN = {FAST_RUN}")

## 1. Tensors

### 1.1 What a tensor is

A `torch.Tensor` is a multi-dimensional array of numbers of a single **dtype**, stored on a **device**
(CPU or GPU), with a **shape** and optionally a record of the computation that produced it. In the
mathematical sense a tensor generalises scalars (0-D), vectors (1-D) and matrices (2-D) to any number of
dimensions; in practice we use the word for the container and think in terms of shapes such as
$(\text{batch}, \text{features})$ for tabular data, $(\text{batch}, \text{channels}, \text{height}, \text{width})$ for
images and $(\text{batch}, \text{time}, \text{features})$ for sequences. If you know NumPy, most of the
API will feel familiar — `torch.zeros`, `torch.arange`, `x.sum(dim=0)`, `x @ y` — and where PyTorch differs
it is usually for one of two reasons: the GPU, or autograd.

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])      # from nested Python lists
b = torch.zeros(2, 3)                                    # shape given as separate ints ...
c = torch.ones((2, 3))                                   # ... or as a tuple
d = torch.arange(6).reshape(2, 3)                        # 0..5 as a 2x3 matrix (integer dtype!)
e = torch.linspace(0, 1, steps=5)                        # 5 evenly spaced numbers in [0, 1]
f = torch.randn(2, 3)                                    # standard normal samples
g = torch.full((2, 3), 7.5)                              # constant tensor
h = torch.eye(3)                                         # identity matrix
r = torch.rand(2, 3)                                     # uniform samples in [0, 1)

for name, t in [("a", a), ("d", d), ("e", e), ("h", h)]:
    print(f"{name}: shape={tuple(t.shape)} dtype={t.dtype} ndim={t.ndim} numel={t.numel()} device={t.device}")
print("a =\n", a)

### 1.2 dtypes

Every tensor has a dtype, and the default for floating-point data is **float32** — not float64 as in NumPy.
This matters: float32 is what GPUs are fast at and what almost all deep learning is done in, but it has only
about seven significant decimal digits, which is why the gradient checks in this notebook switch to
float64 explicitly. Integers default to int64. PyTorch also offers **float16** and **bfloat16** for mixed
precision training (Notebook 14 · Efficient Training and Scaling): float16 has more mantissa bits but a
tiny exponent range (its largest value is 65,504), while bfloat16 keeps float32's range at the price of only
about two to three significant digits. Operations between tensors of different dtypes follow **type promotion** rules
(int + float gives float, float32 + float64 gives float64), and you convert explicitly with `.to(dtype)` or
the shortcuts `.float()`, `.double()`, `.long()`.

In [ ]:
print("default float dtype:", torch.tensor([1.0]).dtype, "| default int dtype:", torch.tensor([1]).dtype)
for dt in [torch.float16, torch.bfloat16, torch.float32, torch.float64]:
    info = torch.finfo(dt)
    print(f"{str(dt):15s} bits={info.bits:2d} max={info.max:.3e} smallest normal={info.tiny:.3e} eps={info.eps:.3e}")

x = torch.tensor(0.1)
print("\n0.1 stored in float32:", f"{x.item():.12f}", "| in float16:", f"{x.half().item():.12f}",
      "| in bfloat16:", f"{x.bfloat16().item():.12f}")
print("type promotion: int64 + float32 ->", (torch.arange(3) + torch.ones(3)).dtype,
      "| float32 * float64 ->", (torch.ones(3) * torch.ones(3, dtype=torch.float64)).dtype)
print("explicit conversion:", torch.arange(3).float().dtype, torch.arange(3).to(torch.float64).dtype)

### 1.3 Indexing and slicing

Indexing follows NumPy: integer indices remove a dimension, slices keep it, `...` stands for "all remaining
dimensions", negative indices count from the end, and a boolean tensor of the same shape selects elements
(**masking**). A tensor of integers used as an index selects rows (**fancy indexing**), which we use
constantly, for instance to pick the predicted probability of each example's true class. `torch.where`
chooses element-wise between two tensors according to a condition, and `torch.gather` collects along a
dimension with a per-row index — both are the vectorised replacements for loops with `if`.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("x =\n", x)
print("x[1]        ->", x[1], "        (row 1, dimension removed)")
print("x[:, 1]     ->", x[:, 1], "          (column 1)")
print("x[1:, ::2]  ->\n", x[1:, ::2], "      (rows 1.., every second column)")
print("x[-1, -1]   ->", x[-1, -1].item(), "                (last element as a Python number)")
print("x[x > 6]    ->", x[x > 6], "      (boolean mask flattens)")
rows = torch.tensor([0, 2, 2]); cols = torch.tensor([1, 0, 3])
print("x[rows, cols] ->", x[rows, cols], "               (fancy indexing picks (0,1), (2,0), (2,3))")
print("torch.where(x % 2 == 0, x, -1) ->\n", torch.where(x % 2 == 0, x, -1))

# Selecting the true-class probability of each example: the idiom used inside every cross-entropy.
probs = torch.tensor([[0.7, 0.2, 0.1], [0.1, 0.1, 0.8]])
targets = torch.tensor([0, 2])
print("probs[arange(N), targets] ->", probs[torch.arange(2), targets])
print("same with gather          ->", probs.gather(1, targets.unsqueeze(1)).squeeze(1))

### 1.4 Views versus copies

A tensor is a *view* onto a block of memory (its **storage**) described by an offset and **strides** — the
number of elements to skip per step along each dimension. Many operations return a new tensor that shares
the same storage with different strides: `view`, `reshape` (when possible), `transpose`/`.t()`, `permute`,
slicing, `squeeze`/`unsqueeze`, `expand`. They cost nothing, but modifying the view modifies the original.
A tensor is **contiguous** when its elements are laid out in memory in row-major order; transposing makes a
tensor non-contiguous, and `view` refuses to work on such tensors (it cannot express the result with
strides alone), whereas `reshape` silently falls back to a copy. `contiguous()` returns a row-major copy
when the tensor is not already contiguous (and the tensor itself when it is), and `clone()` always copies.

> **Common pitfall.** `y = x.view(...)` followed by an in-place change to `y` silently changes `x` too.
> When you need an independent tensor, call `.clone()`.

In [ ]:
x = torch.arange(6.0)
v = x.view(2, 3)                       # same storage, new shape
print("x.data_ptr() == v.data_ptr():", x.data_ptr() == v.data_ptr(), "| strides of v:", v.stride())
v[0, 0] = 100.0
print("after v[0, 0] = 100, x =", x, "   <- the view modified the original")

t = v.t()                              # transpose: strides swapped, no data moved
print("v.t() contiguous?", t.is_contiguous(), "| strides:", t.stride())
try:
    t.view(-1)
except RuntimeError as err:
    print("t.view(-1) fails:", str(err).split(".")[0])
print("t.reshape(-1) works (it copies when it must):", t.reshape(-1))
print("t.contiguous().view(-1) works too:        ", t.contiguous().view(-1))

c = x.clone(); c[0] = -1.0
print("clone is independent: x[0] =", x[0].item(), "c[0] =", c[0].item())

### 1.5 In-place operations

Methods ending in an underscore — `add_`, `mul_`, `zero_`, `clamp_`, `copy_` — modify a tensor in place
instead of allocating a new one, and so do index assignments such as `x[0] = 1` and augmented assignments
such as `x += 1`. In-place operations save memory and are exactly what an optimizer wants when it updates
parameters. They have a cost, though: autograd needs the original values of many tensors to compute
gradients, so it tracks a **version counter** on every tensor and raises an error when a tensor needed for
the backward pass has been modified in place. Section 4.8 provokes this error deliberately; for now,
simply be aware of the two styles.

In [ ]:
x = torch.ones(3)
y = x + 1                    # out of place: new tensor, x unchanged
x.add_(1)                    # in place: x itself changes
x += 1                       # also in place
x[0] = 10.0                  # also in place
print("x =", x, "| y =", y, "| x._version =", x._version, "(bumped by every in-place op)")

### 1.6 Broadcasting

**Broadcasting** lets operations combine tensors of different shapes without copying data. The rule, read
from the *right*: align the shapes at their trailing dimensions; two sizes are compatible if they are equal
or if one of them is 1; a missing dimension counts as 1. The result has the larger size in every position,
and the size-1 (or missing) dimensions are stretched virtually. A few worked examples:

| shapes | aligned | result | example |
|---|---|---|---|
| $(3,)$ and $(1,)$ | $(3), (1)$ | $(3,)$ | scalar times vector |
| $(N, d)$ and $(d,)$ | $(N, d), (1, d)$ | $(N, d)$ | add a bias to every row |
| $(3, 1)$ and $(1, 4)$ | $(3, 1), (1, 4)$ | $(3, 4)$ | outer product / pairwise table |
| $(2, 3, 4)$ and $(4,)$ | $(2,3,4), (1,1,4)$ | $(2, 3, 4)$ | scale the last axis |
| $(3,)$ and $(4,)$ | $(3), (4)$ | **error** | 3 and 4 are neither equal nor 1 |

The most common broadcasting bug is silent, not loud: adding a $(N,)$ vector to a $(N, 1)$ column produces
an $(N, N)$ matrix instead of an error. Whenever two tensors *should* have the same shape, assert it.

In [ ]:
N, d = 4, 3
X = torch.randn(N, d); bias = torch.randn(d)
print("(N, d) + (d,)   ->", (X + bias).shape, "   the bias is added to every row")
col = torch.arange(3.0).reshape(3, 1); row = torch.arange(4.0).reshape(1, 4)
print("(3, 1) * (1, 4) ->", (col * row).shape, "   an outer product:\n", col * row)
print("(2, 3, 4) * (4,) ->", (torch.ones(2, 3, 4) * torch.arange(4.0)).shape)
print("torch.broadcast_shapes((2, 1, 4), (3, 1)) ->", torch.broadcast_shapes((2, 1, 4), (3, 1)))
try:
    torch.ones(3) + torch.ones(4)
except RuntimeError as err:
    print("(3,) + (4,) -> RuntimeError:", str(err)[:70], "...")

# The silent bug: a column vector minus a row vector is a matrix, not a vector of differences.
y_true, y_pred = torch.randn(N, 1), torch.randn(N)
print("(N, 1) - (N,) has shape", (y_true - y_pred).shape, "<- probably not what you meant")

### 1.7 Reductions with `dim` and `keepdim`

Reductions (`sum`, `mean`, `max`, `min`, `argmax`, `std`, `prod`, `norm`, `logsumexp`) collapse one or more
dimensions. `dim=k` removes dimension $k$; `keepdim=True` keeps it with size 1, which is what you want when
the result will be broadcast back against the original — the row-wise maximum subtracted in a stable
softmax is the canonical example. `max` and `min` along a dimension return a named tuple of values *and*
indices. A useful mental model: `x.sum(dim=0)` sums *over* rows, so the result has one entry per column.

In [ ]:
x = torch.tensor([[1.0, 5.0, 3.0], [4.0, 2.0, 6.0]])
print("x.sum()        ->", x.sum().item(), "         (all elements, a 0-D tensor)")
print("x.sum(dim=0)   ->", x.sum(dim=0), "   (sum over rows: one number per column)")
print("x.sum(dim=1)   ->", x.sum(dim=1), "       (sum over columns: one number per row)")
print("x.mean(dim=1, keepdim=True).shape ->", x.mean(dim=1, keepdim=True).shape)
values, indices = x.max(dim=1)
print("x.max(dim=1)   -> values", values, "indices", indices)
print("x.argmax(dim=1) ->", x.argmax(dim=1))

# keepdim in action: a stable row-wise softmax that broadcasts correctly
z = torch.tensor([[1000.0, 0.0, -1000.0], [1.0, 2.0, 3.0]])
stable = torch.exp(z - z.max(dim=1, keepdim=True).values)
stable = stable / stable.sum(dim=1, keepdim=True)
print("stable softmax:\n", stable, "\nmatches torch.softmax:", torch.allclose(stable, torch.softmax(z, dim=1)))

### 1.8 `einsum`, `stack` and `cat`, and other reshaping tools

`torch.einsum` expresses any product-and-sum over indices in one string: repeated indices are multiplied
and summed, indices that appear only on the input side are summed out, and the output indices are listed
after `->`. It is the clearest way to write batched matrix products and the attention scores of
Notebook 09 · Attention and Transformers, and it removes a whole class of transpose-and-reshape bugs.
`torch.stack` joins tensors along a *new* dimension (all inputs must have the same shape), `torch.cat`
joins along an *existing* one; `chunk` and `split` do the reverse. `unsqueeze`/`squeeze` add or remove
size-1 dimensions, `permute` reorders dimensions, and `flatten` merges a range of them.

In [ ]:
A = torch.randn(2, 3, 4); B = torch.randn(2, 4, 5)         # a batch of 2 matrix pairs
batched = torch.einsum("bij,bjk->bik", A, B)               # batched matmul
print("einsum batched matmul == torch.bmm:", torch.allclose(batched, torch.bmm(A, B)), batched.shape)
M = torch.randn(4, 4)
print("trace via einsum:", torch.einsum("ii->", M).item(), "== torch.trace:", M.trace().item())
u, v = torch.randn(3), torch.randn(5)
print("outer product via einsum:", torch.einsum("i,j->ij", u, v).shape)
Q, K = torch.randn(2, 6, 8), torch.randn(2, 7, 8)          # (batch, queries, dim), (batch, keys, dim)
print("attention scores 'bqd,bkd->bqk':", torch.einsum("bqd,bkd->bqk", Q, K).shape)

t1, t2 = torch.zeros(3), torch.ones(3)
print("stack -> new dim:", torch.stack([t1, t2]).shape, "| cat -> existing dim:", torch.cat([t1, t2]).shape)
img = torch.randn(3, 32, 32)                                # (channels, height, width)
print("unsqueeze(0):", img.unsqueeze(0).shape, "| permute to (H, W, C):", img.permute(1, 2, 0).shape,
      "| flatten:", img.flatten().shape, "| chunk into 3:", [c.shape for c in img.chunk(3, dim=0)][0])

### 1.9 NumPy interoperability

`torch.from_numpy(arr)` wraps a NumPy array *without copying*: the tensor and the array share memory, and
the tensor inherits the array's dtype — usually float64, which is rarely what you want for training, so
convert with `.float()`. In the other direction, `tensor.numpy()` also shares memory but works only for
CPU tensors that do not require gradients; the idiom for anything that might live on a GPU or inside a
graph is `tensor.detach().cpu().numpy()`, and it is the form we use before every call to matplotlib.
`torch.as_tensor` and `torch.tensor` also accept arrays; `torch.tensor` always copies.

In [ ]:
arr = np.arange(4.0)                                     # float64 in NumPy
t = torch.from_numpy(arr)
print("from_numpy dtype:", t.dtype, "| shares memory:", t.data_ptr() == arr.ctypes.data)
arr[0] = 99.0
print("after arr[0] = 99, tensor =", t, "   <- shared memory")

t32 = torch.from_numpy(arr).float()                      # converting makes a copy in float32
tg = t32.to(device)                                      # possibly on the GPU
back = tg.detach().cpu().numpy()                         # the safe way back to NumPy
print("round trip dtype:", back.dtype, "| values:", back)

### 1.10 Devices

A tensor lives on one device, and operations require all their operands to be on the same device — mixing
a CPU tensor with a GPU tensor raises an error. The two idioms are `x.to(device)` (returns a copy on the
target, or `x` itself if it is already there) and creating tensors directly on the device with
`device=device`. GPU work is **asynchronous**: `y = x @ x` returns immediately while the GPU is still
computing, so timings must call `torch.cuda.synchronize()` before reading the clock. The benchmark below
runs on whatever hardware is available; with a GPU you should see a speed-up of one to two orders of
magnitude for large matrix products, and essentially none for tiny ones, where launch overhead dominates.

In [ ]:
def time_matmul(n, dev, repeats=5):
    """Average seconds for one n x n matrix product on device dev (synchronised)."""
    x = torch.randn(n, n, device=dev)
    _ = x @ x                                             # warm-up (allocations, kernel loading)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        _ = x @ x
    if dev.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats


sizes = [64, 512, 1024] if FAST_RUN else [64, 512, 2048, 4096]
print(f"{'n':>6s} {'cpu (ms)':>10s} {'gpu (ms)':>10s} {'speed-up':>9s}")
for n in sizes:
    cpu_t = time_matmul(n, torch.device("cpu"))
    if device.type == "cuda":
        gpu_t = time_matmul(n, device)
        print(f"{n:6d} {cpu_t * 1e3:10.2f} {gpu_t * 1e3:10.2f} {cpu_t / gpu_t:9.1f}x")
    else:
        print(f"{n:6d} {cpu_t * 1e3:10.2f} {'n/a':>10s} {'(no GPU)':>9s}")

if device.type == "cuda":
    try:
        torch.ones(2, device=device) + torch.ones(2)          # GPU tensor + CPU tensor
    except RuntimeError as err:
        print("mixing devices ->", str(err)[:80], "...")
else:
    print("No GPU in this session: every tensor is on the CPU, so device mixing cannot be demonstrated.")

On a CPU the time grows roughly with $n^3$, as expected for a dense matrix product. On a GPU the small
products are no faster (the fixed cost of launching a kernel and synchronising is a few tens of
microseconds) while the large ones are dramatically faster — this is the whole reason deep learning runs on
GPUs, and why we always process data in batches: one product of a $(512, 784)$ batch with a $(784, 256)$
weight matrix uses the hardware far better than 512 matrix-vector products.

> **Key idea.** Tensors are NumPy arrays with three extras: a device, a strict default of float32, and the
> ability to record how they were computed. Everything else — shapes, broadcasting, views — you already know.

## 2. Computational graphs and reverse-mode automatic differentiation

### 2.1 Three ways to compute a derivative

There are three ways to obtain the derivative of a function written as code. **Symbolic** differentiation
manipulates the formula (as a computer algebra system does) and produces another formula; for a neural
network this expression explodes in size. **Numerical** differentiation uses finite differences; it needs
at least one extra function evaluation *per input dimension* (two for central differences), which for a
million parameters means a million forward passes, and it is inexact. **Automatic differentiation (AD)** takes the middle road: it evaluates the
function numerically, but at the same time applies the chain rule to every elementary operation, producing
exact derivatives (to floating-point precision) at a cost proportional to the cost of the function itself.
Backpropagation is a special case of AD.

### 2.2 The computational graph

AD views a computation as a **directed acyclic graph** whose nodes are intermediate values and whose edges
are elementary operations with known local derivatives. Take

$$
L = (a \cdot b + c) \cdot d .
$$

Introducing names for the intermediates, $e = a b$, $f = e + c$, $L = f d$, the graph is

```
a ──┐
    (×)── e ──┐
b ──┘         (+)── f ──┐
c ────────────┘         (×)── L
d ──────────────────────┘
```

Every node knows its **local derivatives** with respect to its direct inputs: for $e = ab$ they are
$\partial e/\partial a = b$ and $\partial e/\partial b = a$; for $f = e + c$ they are both 1; for $L = fd$ they are
$d$ and $f$. The chain rule assembles the global derivative $\partial L/\partial a$ from the local ones along
every path from $a$ to $L$: $\partial L/\partial a = \partial L/\partial f \cdot \partial f/\partial e \cdot \partial e/\partial a = d \cdot 1 \cdot b$.
If a variable reaches the output along several paths — as $x$ does in $x \cdot x$, or as every weight does
in a network with shared parameters — the contributions of all paths are **added**.

### 2.3 Forward mode versus reverse mode

The chain rule can be evaluated in two orders. **Forward mode** carries, alongside every value, its
derivative with respect to *one chosen input*: starting from $\dot{a} = 1, \dot{b} = \dot{c} = \dot{d} = 0$ it
propagates $\dot{e} = \dot{a} b + a \dot{b}$, $\dot{f} = \dot{e} + \dot{c}$, $\dot{L} = \dot{f} d + f \dot{d}$ and
ends with $\partial L/\partial a$. One sweep gives the derivative of *all outputs* with respect to *one input*;
for $n$ inputs we need $n$ sweeps. **Reverse mode** does the opposite: it first runs the forward pass and
stores every intermediate, then sweeps from the output back to the inputs carrying $\bar{v} = \partial L / \partial v$
for every node $v$ — the **adjoint** or upstream gradient of Notebook 02 · Neural Networks from Scratch (NumPy).
Starting from $\bar{L} = 1$,
each node passes to each of its inputs the product of its adjoint with the corresponding local derivative,
and inputs reached several times accumulate the sum:

$$
\bar{f} = \bar{L}\, d, \quad \bar{d} = \bar{L}\, f, \quad \bar{e} = \bar{f}, \quad \bar{c} = \bar{f}, \quad \bar{a} = \bar{e}\, b, \quad \bar{b} = \bar{e}\, a .
$$

One reverse sweep gives the derivative of *one output* with respect to *all inputs*. A neural network has
one scalar output (the loss) and millions of inputs (the parameters), so reverse mode computes the entire
gradient in a single backward pass costing a small constant multiple of the forward pass (about twice, in
practice) — exactly the two-matrix-products-per-layer accounting of Notebook 02. Forward mode would need
one pass per parameter. The price of
reverse mode is memory: every intermediate value must be kept until the backward pass has used it, which is
why training uses far more memory than inference and why frameworks free the graph as soon as `backward()`
has run.

The next cell evaluates the example graph by hand in reverse mode and confirms the numbers against
finite differences; Section 3 then turns the procedure into code.

In [ ]:
a, b, c, d = 2.0, -3.0, 10.0, 0.5
e = a * b; f = e + c; L = f * d                       # forward pass, keep the intermediates

L_bar = 1.0                                           # reverse pass, from the output backwards
f_bar = L_bar * d; d_bar = L_bar * f
e_bar = f_bar * 1.0; c_bar = f_bar * 1.0
a_bar = e_bar * b; b_bar = e_bar * a
print(f"L = {L:.2f}")
print(f"reverse mode: dL/da = {a_bar:.3f}  dL/db = {b_bar:.3f}  dL/dc = {c_bar:.3f}  dL/dd = {d_bar:.3f}")

def L_of(a, b, c, d):
    return (a * b + c) * d
eps = 1e-6
fd = [(L_of(*(v + eps * (i == j) for j, v in enumerate([a, b, c, d]))) -
       L_of(*(v - eps * (i == j) for j, v in enumerate([a, b, c, d])))) / (2 * eps) for i in range(4)]
print("finite differences:  " + "  ".join(f"{g:.3f}" for g in fd))

## 3. A scalar autograd engine

### 3.1 Design

We now build reverse-mode AD in under a hundred lines of Python. The design follows micrograd (Karpathy,
2020): a class `Value` wraps a single floating-point number and, in addition to its `data`, stores

- `grad`: the adjoint $\partial L / \partial v$, initialised to 0 and filled in by `backward()`;
- `_prev`: the set of `Value`s that were used to compute this one (the graph edges);
- `_backward`: a closure that, given this node's `grad`, adds the appropriate contribution to each
  input's `grad` — the local chain-rule step of Section 2.3;
- `_op`: a label for the operation, useful when printing the graph.

Every arithmetic operator (`+`, `*`, `**`, unary minus, division) and every function (`exp`, `log`,
`tanh`, `relu`) creates a new `Value`, records the inputs, and attaches a `_backward` that encodes the
operation's local derivative. Nothing is differentiated when the forward pass runs; we merely record
*what would be needed*. The whole engine is the pairing of "how to compute this" with "how to push a
gradient through it", which is precisely the `forward`/`backward` pair of Notebook 02 · Neural Networks from
Scratch (NumPy) reduced to scalars.

Two details are worth noting before the code. First, the backward closures use `+=`, never `=`: a node
that feeds into several others receives several contributions, and the chain rule says they add. Second,
Python operators with a plain number on the left (`2 * v`) call the *reflected* methods `__radd__`,
`__rmul__`, `__rsub__`, `__rtruediv__`, so we implement those too by wrapping the number in a `Value`.

In [ ]:
class Value:
    """A scalar with a recorded computational graph and reverse-mode autodiff."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ---- arithmetic -------------------------------------------------------------------------
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():                      # d(out)/d(self) = 1, d(out)/d(other) = 1
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():                      # d(out)/d(self) = other, d(out)/d(other) = self
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents are supported"
        out = Value(self.data ** exponent, (self,), f"**{exponent}")

        def _backward():                      # d(x^n)/dx = n x^(n-1)
            self.grad += exponent * self.data ** (exponent - 1) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        return self * other ** -1.0

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return Value(other) - self

    def __rtruediv__(self, other):
        return Value(other) * self ** -1.0

    # ---- functions --------------------------------------------------------------------------
    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")

        def _backward():                      # d(e^x)/dx = e^x = out
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")

        def _backward():                      # d(log x)/dx = 1/x
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():                      # d(tanh x)/dx = 1 - tanh^2 x
            self.grad += (1.0 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def _backward():                      # d(relu x)/dx = 1[x > 0]
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    # ---- reverse-mode sweep -----------------------------------------------------------------
    def backward(self):
        """Compute d(self)/d(node) for every node in the graph below self."""
        topo, visited = [], set()

        def build(v):                          # depth-first post-order = topological order
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        for v in topo:                         # intermediate nodes get a fresh adjoint on every sweep;
            if v._prev:                        # leaves (inputs, parameters) accumulate, as in PyTorch
                v.grad = 0.0
        self.grad = 1.0                        # dL/dL = 1
        for v in reversed(topo):               # visit every node after all nodes that depend on it
            v._backward()


a, b, c, d = Value(2.0), Value(-3.0), Value(10.0), Value(0.5)
L = (a * b + c) * d
L.backward()
print("L =", L.data, "| grads:", a.grad, b.grad, c.grad, d.grad, "(compare Section 2.3)")

The gradients match the hand computation and the finite differences of Section 2.3.

### 3.2 Why a topological sort?

A node's `_backward` uses `out.grad`, so it may only run once `out.grad` is *complete* — after every node
that depends on `out` has added its contribution. Calling the closures in an arbitrary order would use
partial adjoints and give wrong answers. A **topological order** of a DAG lists every node after all of
its inputs; walking it in *reverse* therefore lists every node after everything that depends on it, which
is exactly the guarantee we need. The depth-first `build` function produces such an order: it appends a
node only after recursing into all of its children (a post-order traversal), and the `visited` set makes
sure that a node used several times — think of a weight shared across time steps in a recurrent network —
appears exactly once. `backward()` then seeds the output with $\bar{L} = 1$ and runs the closures in
reverse topological order. This is the entire algorithm; `torch.autograd` does the same thing with tensors
instead of scalars, in C++, with many engineering refinements.

One small refinement is built in: before the sweep, `backward()` resets the `grad` of every *intermediate*
node (every node with inputs) to zero, so that only the **leaves** — inputs and parameters, the nodes with
no `_prev` — carry their gradient from one call to the next. Without this, a second sweep would add its
contributions on top of stale intermediate adjoints and produce wrong numbers. PyTorch makes the same
distinction: non-leaf gradients are recomputed on every backward pass and only leaf `.grad` attributes
persist (Section 4.3).

A consequence worth internalising: leaf gradients **accumulate**. Calling `backward()` twice without
resetting `grad` to zero doubles every leaf gradient, because each closure adds to whatever is already
there. This is a feature — it is how contributions from multiple paths combine — but it means that a
training loop must zero the parameter gradients before every backward pass. PyTorch inherits exactly this
behaviour (Section 4.2).

In [ ]:
x = Value(3.0)
y = x * x + x            # x reaches y along two paths: dy/dx = 2x + 1 = 7
y.backward()
print("after one backward:  x.grad =", x.grad)
y.backward()
print("after a second one:  x.grad =", x.grad, "  <- accumulated, not recomputed")
x.grad = 0.0
y.backward()
print("after zeroing first: x.grad =", x.grad)

### 3.3 Verification against finite differences and against PyTorch

Before building anything on the engine we verify it the way Notebook 02 verified every backward pass. We
take a function that exercises every operation — addition, multiplication, subtraction, division, powers,
`exp`, `log`, `tanh`, `relu` — evaluate its gradient with `Value.backward()`, and compare with central
differences computed with plain floats and with `torch.autograd` on the same expression.

In [ ]:
def messy(x, y, z):
    """A function using every operation of the engine. Works on floats, Values and tensors."""
    p = (x * y + z) ** 2 - x / (y - 0.5)
    q = (p * 0.1).exp() + (z ** 2 + 1.0).log() * y
    return q.tanh() * 3.0 + (x - z).relu() + 2.0 / y


# engine gradient
vx, vy, vz = Value(1.3), Value(-0.7), Value(2.1)
out = messy(vx, vy, vz)
out.backward()
engine = [vx.grad, vy.grad, vz.grad]

# finite differences on floats (math functions attached via a tiny wrapper class)
class Float(float):
    def exp(self): return Float(math.exp(self))
    def log(self): return Float(math.log(self))
    def tanh(self): return Float(math.tanh(self))
    def relu(self): return Float(max(0.0, self))
    def __add__(self, o): return Float(float(self) + float(o))
    def __radd__(self, o): return Float(float(o) + float(self))
    def __sub__(self, o): return Float(float(self) - float(o))
    def __rsub__(self, o): return Float(float(o) - float(self))
    def __mul__(self, o): return Float(float(self) * float(o))
    def __rmul__(self, o): return Float(float(o) * float(self))
    def __truediv__(self, o): return Float(float(self) / float(o))
    def __rtruediv__(self, o): return Float(float(o) / float(self))
    def __pow__(self, o): return Float(float(self) ** float(o))

eps, point = 1e-6, [1.3, -0.7, 2.1]
numeric = []
for i in range(3):
    plus = [Float(v + (eps if j == i else 0.0)) for j, v in enumerate(point)]
    minus = [Float(v - (eps if j == i else 0.0)) for j, v in enumerate(point)]
    numeric.append((messy(*plus) - messy(*minus)) / (2 * eps))

# torch.autograd on the same expression (float64 for a fair comparison)
tx, ty, tz = (torch.tensor(v, dtype=torch.float64, requires_grad=True) for v in point)
messy(tx, ty, tz).backward()
torch_grads = [tx.grad.item(), ty.grad.item(), tz.grad.item()]

print(f"{'':10s} {'d/dx':>12s} {'d/dy':>12s} {'d/dz':>12s}")
for name, g in [("engine", engine), ("numeric", numeric), ("torch", torch_grads)]:
    print(f"{name:10s} " + " ".join(f"{v:12.6f}" for v in g))
print("engine vs torch max abs diff:", max(abs(e - t) for e, t in zip(engine, torch_grads)))

All three agree to the six printed decimals; the engine and `torch.autograd` in fact agree to machine
precision, while the finite-difference column is limited to roughly eight digits by the step size and
floating-point cancellation. Note how the same Python function `messy` ran on three
different kinds of objects — floats, `Value`s and tensors — because all three implement the same
operators. This is the pattern PyTorch relies on: the code that defines a model is ordinary Python
arithmetic, and the framework's objects record the graph as a side effect of executing it.

### 3.4 A tiny MLP on the engine

To show that the engine really trains networks, we build the classes of Notebook 02 on top of `Value`.
A `Neuron` holds a list of `Value` weights and a bias, computes $\tanh(\mathbf{w}^\top \mathbf{x} + b)$ (or the
raw affine output for the last layer), and reports its parameters. A `Layer` is a list of neurons, an
`MLP` a list of layers. Weights are drawn uniformly from $[-1, 1]$ and scaled by $1/\sqrt{\text{fan-in}}$, a
simple rule that keeps the tanh units away from saturation (Notebook 04 · Training Deep Networks:
Optimization, Initialization, Normalization, Regularization treats initialization properly).

There is no matrix multiplication anywhere: every multiply-add is a separate `Value`. That is what makes
the engine easy to understand — and, as we will measure, hopelessly slow.

In [ ]:
class Neuron:
    def __init__(self, n_in, nonlin=True, rng=None):
        scale = 1.0 / math.sqrt(n_in)
        self.w = [Value(rng.uniform(-1, 1) * scale) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((w_i * x_i for w_i, x_i in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out, nonlin=True, rng=None):
        self.neurons = [Neuron(n_in, nonlin, rng) for _ in range(n_out)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, n_in, sizes, rng):
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1], nonlin=(i < len(sizes) - 1), rng=rng)
                       for i in range(len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


def count_nodes(root):
    """Number of Value nodes in the graph below root."""
    seen, stack = set(), [root]
    while stack:
        v = stack.pop()
        if v not in seen:
            seen.add(v); stack.extend(v._prev)
    return len(seen)


rng = np.random.default_rng(SEED)
HIDDEN = [8, 8, 1] if FAST_RUN else [16, 16, 1]
value_mlp = MLP(2, HIDDEN, rng)
print(f"MLP 2-{'-'.join(map(str, HIDDEN))}: {len(value_mlp.parameters())} parameters")
out = value_mlp([Value(0.5), Value(-1.0)])
print(f"one forward pass creates a graph of {count_nodes(out)} Value nodes for a single input")

The training problem is the two-moons dataset again, with targets encoded as $-1$ and $+1$ so that the
tanh output can be scored with a squared error: $L = \frac{1}{N} \sum_n (\hat{y}_n - y_n)^2$. (A
cross-entropy would need a `sigmoid` or a softmax in the engine; Exercises 1 and 2 add them.) One training
step is the familiar sequence — forward, zero gradients, backward, update — written directly on `Value`s.
The budget is small on purpose: the engine handles one scalar at a time, so even this toy problem needs a
sizeable fraction of a second per step.

In [ ]:
N_POINTS = 60 if FAST_RUN else 100
X_np, y_np = make_moons(n_samples=N_POINTS, noise=0.1, random_state=SEED)
X_list = X_np.tolist()
y_pm = [2.0 * float(t) - 1.0 for t in y_np]                     # labels in {-1, +1}


def value_mlp_step(model, X, y, lr):
    """One full-batch gradient step on the Value MLP. Returns (loss, accuracy)."""
    outs = [model([Value(x1), Value(x2)]) for x1, x2 in X]
    loss = sum(((o - t) ** 2 for o, t in zip(outs, y)), Value(0.0)) * (1.0 / len(X))
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    for p in model.parameters():
        p.data -= lr * p.grad
    acc = np.mean([(o.data > 0) == (t > 0) for o, t in zip(outs, y)])
    return loss.data, acc


STEPS = 60 if FAST_RUN else 100
losses = []
t0 = time.time()
for step in range(1, STEPS + 1):
    loss_val, acc = value_mlp_step(value_mlp, X_list, y_pm, lr=0.5)
    losses.append(loss_val)
    if step % max(1, STEPS // 5) == 0 or step == 1:
        print(f"step {step:4d} | loss {loss_val:.4f} | train acc {acc:.3f} | {time.time() - t0:5.1f}s")
seconds_per_step = (time.time() - t0) / STEPS

plt.figure()
plt.plot(losses); plt.xlabel("step"); plt.ylabel("mean squared error"); plt.title("Value-engine MLP on two moons")
plt.tight_layout(); plt.show()

To see what the network learned we evaluate it on a grid over the input plane and shade the region where
the output is positive. The grid is deliberately coarse ($36 \times 36$ points): each grid point is a full
scalar forward pass through the engine, so a fine grid would take longer than the training itself.

In [ ]:
# The decision boundary, evaluated on a coarse grid (each grid point is a full scalar forward pass).
xx, yy = np.meshgrid(np.linspace(X_np[:, 0].min() - 0.4, X_np[:, 0].max() + 0.4, 36),
                     np.linspace(X_np[:, 1].min() - 0.4, X_np[:, 1].max() + 0.4, 36))
zz = np.array([value_mlp([Value(x1), Value(x2)]).data for x1, x2 in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)
plt.figure()
plt.contourf(xx, yy, zz > 0, alpha=0.3, cmap="coolwarm")
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", edgecolors="k", s=35)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.title("decision boundary of the Value-engine MLP")
plt.tight_layout(); plt.show()

The loss falls steadily and the boundary curves around the two moons: a network trained entirely by our
hundred-line engine. Now the cost. The cell below times one forward-and-backward pass of the `Value` MLP
against the same computation on a batched tensor with `torch.autograd`.

In [ ]:
# The same network as a batched tensor computation with torch.autograd.
sizes = [2] + HIDDEN
W_t = [torch.randn(sizes[i], sizes[i + 1], dtype=torch.float64, requires_grad=True) for i in range(len(HIDDEN))]
b_t = [torch.zeros(sizes[i + 1], dtype=torch.float64, requires_grad=True) for i in range(len(HIDDEN))]
X_t = torch.tensor(X_list, dtype=torch.float64); y_t = torch.tensor(y_pm, dtype=torch.float64)


def torch_step_time(repeats=20):
    t0 = time.perf_counter()
    for _ in range(repeats):
        h = X_t
        for i, (W, b) in enumerate(zip(W_t, b_t)):
            h = h @ W + b
            if i < len(W_t) - 1:
                h = torch.tanh(h)
        loss = ((h.squeeze(1) - y_t) ** 2).mean()
        for p in W_t + b_t:
            p.grad = None
        loss.backward()
    return (time.perf_counter() - t0) / repeats


torch_seconds = torch_step_time()
print(f"Value engine : {seconds_per_step * 1e3:8.1f} ms per step ({count_nodes(out) * len(X_list):,} scalar nodes)")
print(f"torch tensors: {torch_seconds * 1e3:8.2f} ms per step (CPU, same network, same batch)")
print(f"ratio        : {seconds_per_step / torch_seconds:8.0f}x slower")

The scalar engine is typically two to three orders of magnitude slower than the tensor version — for a
network with a few hundred parameters and about a hundred data points. The reason is not that
Python is slow at arithmetic (though it is); it is that the engine creates one Python object, one closure
and one small set of parents per multiply-add, and then walks a graph of tens of thousands of nodes for the
backward pass. A tensor library records *one* node per matrix product, no matter how large the matrices,
and the arithmetic itself runs in optimised C++ or CUDA kernels. The engine taught us what autograd does;
tensors are how it is done at scale. From here on we use `torch.autograd`.

> **Key idea.** Reverse-mode AD is: record the graph during the forward pass, then walk it in reverse
> topological order, letting each node multiply the upstream gradient by its local derivative and add the
> result to its inputs. The only difference between micrograd and PyTorch is the granularity of a node.

## 4. `torch.autograd`

### 4.1 `requires_grad`, `backward()` and `.grad`

PyTorch's engine is switched on per tensor by the flag `requires_grad`. Any tensor computed from a tensor
that requires gradients also requires them, and carries a `grad_fn` — the tensor-level counterpart of our
`_backward` closure — pointing to the node that created it. Calling `backward()` on a scalar runs the
reverse sweep and deposits gradients in the `.grad` attribute of every **leaf** tensor with
`requires_grad=True`. A leaf is a tensor that was created directly by the user rather than by an
operation; parameters are leaves, activations are not.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3 + 2 * x                      # dy/dx = 3x^2 + 2 = 14 at x = 2
print("y =", y, "| grad_fn:", y.grad_fn, "| x.is_leaf:", x.is_leaf, "| y.is_leaf:", y.is_leaf)
y.backward()
print("x.grad =", x.grad)

# A more realistic example: the gradient of a loss with respect to a weight matrix.
W = torch.randn(3, 2, requires_grad=True)
X = torch.randn(5, 3)
loss = ((X @ W) ** 2).sum()
loss.backward()
print("W.grad has the shape of W:", W.grad.shape, "| analytic 2 X^T (X W) matches:",
      torch.allclose(W.grad, 2 * X.T @ (X @ W)))

### 4.2 Gradients accumulate — so we zero them

Exactly as in our engine, `.grad` is accumulated, not overwritten. This is deliberate: it allows gradient
accumulation over several mini-batches (Notebook 14 · Efficient Training and Scaling) and it is what makes
shared parameters work. It also
means that a training loop must clear the gradients before each backward pass, either by setting
`p.grad = None` (cheapest; the next backward allocates a fresh tensor) or with `p.grad.zero_()`; with an
optimizer, `optimizer.zero_grad()` does this for every parameter. Forgetting it is the most common
autograd bug in beginner code: the gradients grow with every step and training diverges.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (3 * x).backward()
    print(f"backward #{i + 1}: x.grad = {x.grad.item():.1f}   (true derivative is 3)")
x.grad = None
(3 * x).backward()
print(f"after x.grad = None: x.grad = {x.grad.item():.1f}")

### 4.3 Leaf and non-leaf tensors, `retain_grad` and `retain_graph`

Only leaves keep their `.grad`. For an intermediate tensor (a non-leaf) autograd computes the gradient
during the sweep — it needs it to continue backwards — but discards it afterwards to save memory, and
reading `.grad` returns `None` with a warning. If you do want it (for instance to inspect the gradient of
an activation while debugging), call `.retain_grad()` on the tensor before `backward()`.

A second memory-saving default: `backward()` **frees the graph** once it has been used. The intermediate
values that the local derivatives need (the input of a `tanh`, the operands of a product) are released,
so calling `backward()` a second time on the same graph raises an error. If you genuinely need two sweeps
over one graph — say, to compute gradients of two different losses that share a forward pass — pass
`retain_graph=True` to the first call.

In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
h = x * 3                    # non-leaf
h.retain_grad()              # ask autograd to keep dL/dh
L = (h ** 2).sum()
L.backward(retain_graph=True)
print("x.grad =", x.grad, "| h.grad (retained) =", h.grad, "| h.is_leaf:", h.is_leaf)

L.backward()                 # allowed because the graph was retained; gradients accumulate again
print("after second backward: x.grad =", x.grad)
try:
    L.backward()             # the second call did not retain the graph -> error
except RuntimeError as err:
    print("third backward -> RuntimeError:", str(err)[:95], "...")

### 4.4 Turning autograd off: `torch.no_grad`, `detach` and `requires_grad_`

Recording the graph costs memory and time, and there are three common situations where we do not want it.
During **evaluation** nothing will be differentiated, so the whole forward pass runs under
`with torch.no_grad():` (or the slightly faster `torch.inference_mode()`), which makes every result a
plain tensor without history. During a **parameter update** we modify leaves that require gradients in
place, which autograd forbids inside a recorded region — the update must also happen under `no_grad`.
Finally, when a single tensor should be treated as a constant, `x.detach()` returns a view that shares
the data but is cut off from the graph; it is the tool for target networks in reinforcement learning
(Notebook 13 · Reinforcement Learning), for logging, and for the `.detach().cpu().numpy()` idiom. `requires_grad_()` flips the
flag on an existing leaf in place, e.g. to freeze or unfreeze parameters.

In [ ]:
w = torch.tensor([1.0, -1.0], requires_grad=True)
x = torch.tensor([2.0, 3.0])

with torch.no_grad():
    y_eval = w * x
print("inside no_grad: requires_grad =", y_eval.requires_grad, "| grad_fn =", y_eval.grad_fn)

y = w * x
y_det = y.detach()
print("detach: shares data ->", y_det.data_ptr() == y.data_ptr(), "| requires_grad ->", y_det.requires_grad)

# The parameter update: an in-place change to a leaf is only legal without recording.
try:
    w -= 0.1 * torch.ones(2)
except RuntimeError as err:
    print("update outside no_grad -> RuntimeError:", str(err)[:70], "...")
with torch.no_grad():
    w -= 0.1 * torch.ones(2)
print("update under no_grad works: w =", w)

w.requires_grad_(False)
print("after requires_grad_(False):", (w * x).requires_grad, "  (frozen: no graph is recorded)")
w.requires_grad_(True)

### 4.5 Non-scalar outputs and vector-Jacobian products

`backward()` without arguments works only on a scalar, because "the gradient of a vector" is a matrix — the
Jacobian $J = \partial \mathbf{y} / \partial \mathbf{x}$ of shape $(m, n)$ — and autograd never materialises Jacobians.
What reverse mode actually computes is the **vector-Jacobian product** $\mathbf{v}^\top J$ for a chosen vector
$\mathbf{v} \in \mathbb{R}^m$: `y.backward(v)` deposits $\mathbf{v}^\top J$ in `x.grad`. The scalar case is the special
case $m = 1$, $\mathbf{v} = 1$. The interpretation is the chain rule: if some downstream scalar $L$ had
$\partial L/\partial \mathbf{y} = \mathbf{v}$, then $\partial L/\partial \mathbf{x} = \mathbf{v}^\top J$. Passing
$\mathbf{v} = \mathbf{e}_i$ (a one-hot vector) extracts row $i$ of the Jacobian, so $m$ backward passes recover the whole
matrix — which we do below to check against `torch.func.jacrev`, the function-transform API that does it
efficiently.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = torch.stack([x[0] * x[1], x[1] ** 2 + x[2], x[0] * x[2] * 10])   # y: R^3 -> R^3
try:
    y.backward()
except RuntimeError as err:
    print("y.backward() on a vector ->", str(err)[:60], "...")

v = torch.tensor([1.0, 0.0, 0.0])                     # pick row 0 of the Jacobian
y.backward(v, retain_graph=True)
print("v^T J for v = e_0:", x.grad, "   (dy_0/dx = [x_1, x_0, 0] = [2, 1, 0])")

# The full Jacobian row by row, versus torch.func.jacrev.
def f(x):
    return torch.stack([x[0] * x[1], x[1] ** 2 + x[2], x[0] * x[2] * 10])

J_rows = []
for i in range(3):
    x.grad = None
    y.backward(torch.eye(3)[i], retain_graph=True)
    J_rows.append(x.grad.clone())
J_manual = torch.stack(J_rows)
J_func = torch.func.jacrev(f)(x.detach())
print("Jacobian via 3 VJPs:\n", J_manual, "\nmatches torch.func.jacrev:", torch.allclose(J_manual, J_func))

### 4.6 `torch.autograd.grad` and higher-order derivatives

`backward()` writes into `.grad` attributes; the functional form `torch.autograd.grad(outputs, inputs)`
instead *returns* the gradients, leaving `.grad` untouched, and is the cleaner choice whenever the gradient
is an intermediate result rather than the end of a training step. Its most important option is
`create_graph=True`: the gradient computation is itself recorded, so the returned gradient can be
differentiated again. That gives **higher-order derivatives** — second derivatives for curvature analysis,
gradient penalties (Notebook 12 · GANs and Diffusion Models uses one), and meta-learning.

The cell below differentiates $f(x) = \sin(x)\,e^{-x^2/4}$ twice. Because $f$ is applied element-wise to a
vector of $x$ values, `f(x).sum()` has the gradient $(f'(x_1), \dots, f'(x_n))$ — the sum makes the output a
scalar without mixing the elements — and the same trick works for the second derivative.

In [ ]:
x = torch.linspace(-4, 4, 200, requires_grad=True)
f_x = torch.sin(x) * torch.exp(-x ** 2 / 4)
f1 = torch.autograd.grad(f_x.sum(), x, create_graph=True)[0]      # f'(x), still differentiable
f2 = torch.autograd.grad(f1.sum(), x, create_graph=True)[0]       # f''(x)
f3 = torch.autograd.grad(f2.sum(), x)[0]                          # f'''(x), graph no longer needed

xs = x.detach().cpu().numpy()          # the safe idiom before plotting (x is on the CPU here anyway)
plt.figure()
for curve, label in [(f_x, "$f$"), (f1, "$f'$"), (f2, "$f''$"), (f3, "$f'''$")]:
    plt.plot(xs, curve.detach().cpu().numpy(), label=label)
plt.xlabel("$x$"); plt.title("$f(x) = \\sin(x)\\,e^{-x^2/4}$ and its derivatives via autograd"); plt.legend()
plt.tight_layout(); plt.show()

# analytic check of the first derivative at a few points
f1_analytic = (torch.cos(x) - 0.5 * x * torch.sin(x)) * torch.exp(-x ** 2 / 4)
print("max |autograd f' - analytic f'| =", (f1 - f1_analytic).abs().max().item())
print("x.grad is still None (autograd.grad does not touch it):", x.grad)

The curves have the expected relationships — $f'$ crosses zero at every extremum of $f$, $f''$ at every
inflection point — and the first derivative matches the closed form to float precision. Nothing about the
function was special: any differentiable PyTorch expression can be differentiated repeatedly, and the
cost of each additional order is roughly another forward-and-backward pass.

### 4.7 `torch.func`: Jacobians and Hessians as function transforms

For whole Jacobians and Hessians the loop of Section 4.5 is clumsy. `torch.func` provides composable
**function transforms**: `jacrev(f)` returns a function computing the Jacobian by reverse mode, `jacfwd(f)`
by forward mode, `hessian(f)` composes the two, and `vmap` vectorises over a batch. They take and return
plain functions rather than tensors with history, which makes them natural for per-example gradients — the
saliency maps of Notebook 16 · Interpretability, Robustness and Uncertainty are gradients of one output per
input example — and for second-order quantities such as Hessians. For the quadratic
$f(\mathbf{x}) = \tfrac12 \mathbf{x}^\top A \mathbf{x}$ the Hessian is $\tfrac12 (A + A^\top)$, which we verify.

In [ ]:
A = torch.tensor([[2.0, 1.0, 0.0], [0.0, 3.0, 1.0], [1.0, 0.0, 4.0]])
quad = lambda x: 0.5 * x @ A @ x
x0 = torch.tensor([1.0, -1.0, 2.0])

grad_q = torch.func.grad(quad)(x0)
hess_q = torch.func.hessian(quad)(x0)
print("gradient:", grad_q, "| analytic (A + A^T) x / 2:", 0.5 * (A + A.T) @ x0)
print("Hessian:\n", hess_q, "\nequals (A + A^T)/2:", torch.allclose(hess_q, 0.5 * (A + A.T)))

# vmap: evaluate per-example gradients of a loss for a whole batch in one call
def per_example_loss(w, x, y):
    return ((x @ w - y) ** 2)
w0 = torch.tensor([0.5, -0.5]); xb = torch.randn(4, 2); yb = torch.randn(4)
per_example_grads = torch.func.vmap(torch.func.grad(per_example_loss), in_dims=(None, 0, 0))(w0, xb, yb)
print("per-example gradients shape:", per_example_grads.shape, "(one gradient of w per example)")

### 4.8 In-place operations and the version counter

Section 1.5 mentioned that autograd tracks in-place modifications. Here is the mechanism in action.
`sigmoid` saves its *output* for the backward pass (recall $\sigma' = \sigma(1-\sigma)$). If that output is
then modified in place, the saved tensor no longer holds the right values; rather than silently computing a
wrong gradient, autograd compares the tensor's version counter with the one it recorded and raises an
error naming the operation. The fix is always the same: use the out-of-place variant, or `clone()` first.
A related error occurs when a leaf that requires gradients is modified in place *outside* `no_grad`
(Section 4.4). Exercise 4 gives you a snippet with such a bug to find.

In [ ]:
x = torch.randn(4, requires_grad=True)
s = torch.sigmoid(x)               # saves s for the backward pass
s.mul_(2.0)                        # in-place: bumps the version counter of s
try:
    s.sum().backward()
except RuntimeError as err:
    print("RuntimeError:", str(err)[:120], "...")

s = torch.sigmoid(x)
s2 = s * 2.0                       # out of place: a new tensor, s is untouched
s2.sum().backward()
print("out-of-place version works: x.grad =", x.grad)

### 4.9 A custom `torch.autograd.Function`

Occasionally an operation is not expressible with existing ops, or its default gradient is numerically
poor, or you want a different backward on purpose (straight-through estimators in quantization-aware
training, Notebook 15 · Deployment and Model Optimization). For these cases you subclass
`torch.autograd.Function` and implement two static methods:
`forward(ctx, ...)` computes the output and may stash tensors for later with `ctx.save_for_backward`, and
`backward(ctx, grad_output)` receives the upstream gradient and must return one gradient per input of
`forward` (or `None` for inputs that need none). This is precisely a `Value` operation with its
`_backward` closure, lifted to tensors.

Our example is the **softplus** $\log(1 + e^x)$, whose naive evaluation overflows for large $x$. The stable
form is $\max(x, 0) + \log(1 + e^{-|x|})$, and its derivative is the sigmoid, $\sigma(x)$. We implement it
with a hand-written backward and check it with `torch.autograd.gradcheck`, which compares the custom
backward with finite differences in float64 — the same check as Notebook 02 · Neural Networks from Scratch
(NumPy), provided by the framework.

In [ ]:
class StableSoftplus(torch.autograd.Function):
    """log(1 + exp(x)) computed without overflow, with an explicit backward."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.clamp(x, min=0) + torch.log1p(torch.exp(-x.abs()))

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * torch.sigmoid(x)            # d softplus / dx = sigmoid(x)


softplus = StableSoftplus.apply                          # use the .apply method, not the class

x = torch.tensor([-1000.0, -1.0, 0.0, 1.0, 1000.0], dtype=torch.float64, requires_grad=True)
print("stable softplus  :", softplus(x).detach())
with torch.no_grad():
    print("naive log(1+e^x) :", torch.log(1 + torch.exp(x)), "  <- exp(1000) overflows even in float64")
print("matches F.softplus:", torch.allclose(softplus(x), F.softplus(x)))

x_check = torch.randn(6, dtype=torch.float64, requires_grad=True)
print("gradcheck passes:", torch.autograd.gradcheck(softplus, (x_check,), eps=1e-6, atol=1e-5))

`gradcheck` returns `True`: the hand-written backward agrees with finite differences. Two rules of thumb
for custom functions: always test them with `gradcheck` in float64 (an incorrect backward will not raise
an error — it will just train badly), and prefer composing existing differentiable ops whenever the
result is numerically fine, because then autograd derives the backward for you and it stays correct when
you change the forward.

> **Key idea.** `torch.autograd` is our engine with tensors as nodes. The rules to remember: gradients
> accumulate, so zero them; the graph is freed after `backward()`; wrap evaluation and parameter updates in
> `no_grad`; use `detach()` for constants; `backward(v)` computes $\mathbf{v}^\top J$; `create_graph=True` for
> higher orders; and never modify in place what the backward pass still needs.

## 5. Training with raw tensors and autograd

We now have everything needed to train models without writing a single backward pass. This section
does so deliberately *without* `nn.Module` or `torch.optim`, so that every moving part is visible:
parameters are ordinary tensors with `requires_grad=True`, the forward pass is ordinary arithmetic, the
loss is a scalar tensor, `backward()` fills the `.grad` attributes, and the update is an in-place
subtraction under `no_grad`. Once this pattern is second nature, the abstractions of Section 6 will look
like exactly what they are: bookkeeping.

### 5.1 Linear regression

We fit $y = wx + b$ to noisy data generated from $w = 3$, $b = -2$. The loss is the mean squared error, the
parameters start at zero, and each step is one full-batch gradient descent update. Because this problem
has a closed-form solution (the least-squares normal equations), we can check that gradient descent
converges to it. Every tensor is created on `device`, so the same cell runs unchanged on a CPU or a GPU;
only the plotting needs the `.cpu().numpy()` step of Section 1.9.

In [ ]:
torch.manual_seed(SEED)
N_LIN = 200
x_lin = torch.rand(N_LIN, 1, device=device) * 4 - 2                          # inputs in [-2, 2]
y_lin = 3.0 * x_lin - 2.0 + 0.5 * torch.randn(N_LIN, 1, device=device)       # true w = 3, b = -2, noise std 0.5

w = torch.zeros(1, 1, device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)
LR_LIN, STEPS_LIN = 0.1, 100 if FAST_RUN else 300
loss_history = []
for step in range(STEPS_LIN):
    y_hat = x_lin @ w + b                                      # forward: (N, 1)
    loss = ((y_hat - y_lin) ** 2).mean()                       # scalar loss
    loss.backward()                                            # gradients into w.grad, b.grad
    with torch.no_grad():                                      # update without recording
        w -= LR_LIN * w.grad
        b -= LR_LIN * b.grad
        w.grad = None; b.grad = None                           # reset for the next step
    loss_history.append(loss.item())

X_design = torch.cat([x_lin, torch.ones(N_LIN, 1, device=device)], dim=1)     # closed form: [w, b] = lstsq(X, y)
w_ls, b_ls = torch.linalg.lstsq(X_design, y_lin).solution.flatten().tolist()
print(f"gradient descent: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"least squares   : w = {w_ls:.4f}, b = {b_ls:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(loss_history); axes[0].set_xlabel("step"); axes[0].set_ylabel("MSE"); axes[0].set_title("linear regression: loss")
axes[1].scatter(x_lin.cpu().numpy(), y_lin.cpu().numpy(), s=10, label="data")
xs = torch.linspace(-2, 2, 50, device=device).reshape(-1, 1)
with torch.no_grad():
    axes[1].plot(xs.cpu().numpy(), (xs @ w + b).cpu().numpy(), "r", label="fit")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y"); axes[1].set_title("linear regression: fit"); axes[1].legend()
plt.tight_layout(); plt.show()

Gradient descent reaches the least-squares solution to four decimal places, and the loss curve shows the
characteristic geometric decay of gradient descent on a convex quadratic (Notebook 01 · Mathematical
Foundations, Section 3). The
loop is worth re-reading line by line, because every training loop in PyTorch has the same skeleton:
forward, loss, `backward()`, update under `no_grad`, zero the gradients.

### 5.2 Logistic regression

For binary classification we keep the linear model but interpret its output $z = \mathbf{w}^\top \mathbf{x} + b$
as a **logit**: the probability of class 1 is $\sigma(z)$. The loss is the binary cross-entropy,

$$
L = -\frac{1}{N} \sum_n \big[ y_n \log \sigma(z_n) + (1 - y_n) \log (1 - \sigma(z_n)) \big]
  = \frac{1}{N} \sum_n \big[ \operatorname{softplus}(z_n) - y_n z_n \big],
$$

where the second form follows from $-\log \sigma(z) = \operatorname{softplus}(-z)$ and
$-\log(1 - \sigma(z)) = \operatorname{softplus}(z)$. PyTorch's `F.binary_cross_entropy_with_logits` computes
this stable form (with the same trick as our `StableSoftplus`); never apply a sigmoid and then
`F.binary_cross_entropy` separately. We train on the two moons, which a linear boundary cannot separate,
to make the point that the model — not the training procedure — is what limits us here.

In [ ]:
X_np, y_np = make_moons(n_samples=400, noise=0.2, random_state=SEED)
X_lr = torch.tensor(X_np, dtype=torch.float32, device=device)
y_lr = torch.tensor(y_np, dtype=torch.float32, device=device)

w = torch.zeros(2, device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)
LR_LOG, STEPS_LOG = 0.5, 200 if FAST_RUN else 500
for step in range(1, STEPS_LOG + 1):
    logits = X_lr @ w + b                                      # (N,)
    loss = F.binary_cross_entropy_with_logits(logits, y_lr)
    loss.backward()
    with torch.no_grad():
        w -= LR_LOG * w.grad; b -= LR_LOG * b.grad
        w.grad = None; b.grad = None
    if step % (STEPS_LOG // 4) == 0:
        acc = ((logits > 0).float() == y_lr).float().mean().item()
        print(f"step {step:4d} | BCE {loss.item():.4f} | train acc {acc:.3f}")

# Decision boundary: the line w1 x1 + w2 x2 + b = 0.
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    zz = (grid @ w + b > 0).reshape(xx.shape).cpu().numpy()    # boolean mask, moved to the CPU for matplotlib
plt.figure()
plt.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", edgecolors="k", s=25)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.title(f"logistic regression on two moons (train acc {acc:.2f})")
plt.tight_layout(); plt.show()

Logistic regression finds the best straight line, and the best straight line gets roughly 84% of the
moons right: the tips of both crescents end up on the wrong side. This is XOR all over again. The fix,
as in Notebook 02 · Neural Networks from Scratch (NumPy), is a hidden layer — and with autograd, adding one
costs a single extra line in the
forward pass and nothing in the backward pass. We do that next, at full scale.

### 5.3 The MNIST MLP, rewritten with tensors and autograd

Notebook 02 needed roughly 150 lines — layer classes with hand-derived `backward` methods, a container, an
optimizer, a gradient checker — to train a 784-256-128-10 network on MNIST. Here is the same network with
tensors and autograd. The parameters are six tensors with `requires_grad=True`; the forward pass is three
lines; the loss is `F.cross_entropy` (softmax cross-entropy with log-sum-exp built in, taking logits and
integer labels exactly as our NumPy version did); and the backward pass is `loss.backward()`. We keep the
whole dataset on the device as one tensor and slice mini-batches from a random permutation, which is the
fastest way to feed a model this small; `DataLoader`, which handles datasets that do not fit in memory,
appears in Notebook 05 · The PyTorch Workflow and Best Practices.

The data loader is the one from Notebook 02, returning tensors instead of arrays, with the same offline
fallback to scikit-learn's digits.

In [ ]:
def load_mnist_tensors(data_dir):
    """Return (X_train, y_train, X_test, y_test, name) as CPU tensors with X in [0, 1], flattened."""
    try:
        import torchvision
        train = torchvision.datasets.MNIST(root=str(data_dir), train=True, download=True)
        test = torchvision.datasets.MNIST(root=str(data_dir), train=False, download=True)
        X_tr = train.data.reshape(len(train.data), -1).float() / 255.0
        X_te = test.data.reshape(len(test.data), -1).float() / 255.0
        return X_tr, train.targets.clone(), X_te, test.targets.clone(), "MNIST"
    except Exception as err:
        print(f"Could not load MNIST through torchvision ({err!r}).")
        print("Falling back to sklearn's 8x8 digits so that the notebook still runs offline.")
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        digits = load_digits()
        X_tr, X_te, y_tr, y_te = train_test_split(digits.data / 16.0, digits.target, test_size=0.25,
                                                  random_state=SEED, stratify=digits.target)
        return (torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long),
                torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long), "sklearn digits")


X_train_full, y_train_full, X_test, y_test, DATASET_NAME = load_mnist_tensors(DATA_DIR)
PIXEL_MEAN, PIXEL_STD = X_train_full.mean(), X_train_full.std()
X_train_full = (X_train_full - PIXEL_MEAN) / PIXEL_STD
X_test = (X_test - PIXEL_MEAN) / PIXEL_STD
N_VAL = min(5_000, len(X_train_full) // 5)
X_val, y_val = X_train_full[-N_VAL:], y_train_full[-N_VAL:]
N_TRAIN = min(10_000 if FAST_RUN else 55_000, len(X_train_full) - N_VAL)
X_train, y_train = X_train_full[:N_TRAIN], y_train_full[:N_TRAIN]
D_IN = X_train.shape[1]
print(f"{DATASET_NAME}: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, test {tuple(X_test.shape)}, "
      f"dtype {X_train.dtype}, labels {y_train.dtype}")

Now the model, as four small functions and a training routine. `init_params` creates He-initialised
weight matrices (scaled by $\sqrt{2/\text{fan-in}}$, the rule for ReLU networks derived in Notebook 04 · Training
Deep Networks: Optimization, Initialization, Normalization, Regularization) and
zero biases as leaf tensors with `requires_grad=True`; `mlp_forward` is the three-line forward pass ending
in logits; `sgd_step` is the update under `no_grad` followed by the gradient reset; `accuracy` evaluates
in batches without recording a graph; and `train_mnist` ties them together, re-seeding first so that
repeated runs on the same device are reproducible. We train once on the CPU and, when a GPU is available,
once more on the GPU with the same budget, to compare wall-clock time as well as accuracy.

In [ ]:
def init_params(sizes, dev):
    """He-initialised weight and bias tensors that require gradients, alternating [W1, b1, W2, b2, ...]."""
    params = []
    for fan_in, fan_out in zip(sizes[:-1], sizes[1:]):
        W = (torch.randn(fan_in, fan_out, device=dev) * math.sqrt(2.0 / fan_in)).requires_grad_()
        b = torch.zeros(fan_out, device=dev, requires_grad=True)
        params += [W, b]
    return params


def mlp_forward(params, X):
    """ReLU MLP forward pass; the last layer returns logits."""
    h, n_layers = X, len(params) // 2
    for i in range(n_layers):
        h = h @ params[2 * i] + params[2 * i + 1]
        if i < n_layers - 1:
            h = torch.relu(h)
    return h


def sgd_step(params, lr):
    """In-place SGD update, then reset the gradients."""
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad
            p.grad = None


def accuracy(params, X, y, batch_size=2000):
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            correct += (mlp_forward(params, X[i:i + batch_size]).argmax(dim=1) == y[i:i + batch_size]).sum().item()
    return correct / len(X)


def train_mnist(dev, epochs, batch_size=64, lr=0.1, seed=SEED, verbose=True):
    """Train the 256-128 MLP on device dev. Returns the parameters and the elapsed training time."""
    torch.manual_seed(seed)
    Xtr, ytr, Xva, yva = X_train.to(dev), y_train.to(dev), X_val.to(dev), y_val.to(dev)
    params = init_params([D_IN, 256, 128, 10], dev)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        perm = torch.randperm(len(Xtr), device=dev)
        total = torch.zeros((), device=dev)
        for i in range(0, len(Xtr), batch_size):
            idx = perm[i:i + batch_size]
            loss = F.cross_entropy(mlp_forward(params, Xtr[idx]), ytr[idx])
            loss.backward()
            sgd_step(params, lr)
            total += loss.detach() * len(idx)
        if dev.type == "cuda":
            torch.cuda.synchronize()
        if verbose:
            print(f"epoch {epoch:2d} | train loss {total.item() / len(Xtr):.4f} | "
                  f"val acc {accuracy(params, Xva, yva):.4f} | {time.time() - t0:5.1f}s")
    return params, time.time() - t0


EPOCHS_T = 1 if FAST_RUN else 5
results = {}
for dev in [torch.device("cpu")] + ([device] if device.type == "cuda" else []):
    print(f"--- training on {dev}")
    params, seconds = train_mnist(dev, EPOCHS_T)
    test_acc = accuracy(params, X_test.to(dev), y_test.to(dev))
    results[str(dev)] = (test_acc, seconds)
    print(f"test accuracy {test_acc:.4f} | {seconds:.1f}s for {EPOCHS_T} epoch(s)\n")

if len(results) == 2:
    (acc_cpu, t_cpu), (acc_gpu, t_gpu) = results["cpu"], results[str(device)]
    print(f"GPU speed-up over CPU: {t_cpu / t_gpu:.1f}x (accuracies {acc_cpu:.4f} vs {acc_gpu:.4f})")
else:
    print("No GPU available: the CPU run is the only one. On a GPU this loop typically runs several times faster.")

The accuracy matches Notebook 02 — around 97–98% after five epochs on the full training set, roughly 90%
after one epoch on the `FAST_RUN` subset — because it *is* the same model, the same loss and the same
optimizer; only the gradient computation moved from our hands to autograd. The code that mattered fits in
about forty lines, and none of it would change if we added a fourth layer, swapped ReLU for GELU, or
inserted a skip connection. The CPU run is typically at least as fast as the NumPy version (PyTorch's CPU
kernels are multi-threaded and fuse more work per call); with a GPU the same loop runs several times faster
still, and the gap widens for
larger models and batches, as the matrix-product benchmark of Section 1.10 predicted.

> **Try it.** Add a fourth layer (`[D_IN, 256, 128, 64, 10]`) to `init_params` and retrain. In Notebook 02
> this would have required extending the backward pass; here nothing else changes.

## 6. Preview: `nn.Module` and `torch.optim`

Look at what the raw-tensor version still makes us do by hand: create and initialise parameter tensors,
thread them through a forward function, loop over them for the update, zero the gradients. PyTorch
packages these chores into two abstractions that the rest of the course uses everywhere.

An **`nn.Module`** owns parameters and defines a `forward` method. Built-in modules such as `nn.Linear`
create their own correctly-initialised weights, `nn.Sequential` chains modules, and `model.parameters()`
yields every parameter for the optimizer. Modules also carry a train/eval switch (`model.train()`,
`model.eval()`) that layers such as dropout and batch normalization (Notebook 04 · Training Deep Networks:
Optimization, Initialization, Normalization, Regularization) consult. A
**`torch.optim` optimizer** receives the parameters once and then implements the update rule in
`optimizer.step()` and the gradient reset in `optimizer.zero_grad()`; `SGD`, `SGD(momentum=0.9)`, `Adam`
and `AdamW` all share this interface, so switching optimizers is a one-line change. The cell below trains
the same MNIST network with these tools; compare it line by line with Section 5.3.

In [ ]:
torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(D_IN, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters | state_dict keys: {list(model.state_dict().keys())[:4]} ...")

Xtr, ytr = X_train.to(device), y_train.to(device)
t0 = time.time()
for epoch in range(1, EPOCHS_T + 1):
    model.train()
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), 64):
        idx = perm[i:i + 64]
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
        optimizer.zero_grad()          # instead of p.grad = None for every parameter
        loss.backward()
        optimizer.step()               # instead of p -= lr * p.grad under no_grad
    model.eval()
    with torch.no_grad():
        val_acc = (model(X_val.to(device)).argmax(dim=1) == y_val.to(device)).float().mean().item()
    print(f"epoch {epoch:2d} | last batch loss {loss.item():.4f} | val acc {val_acc:.4f} | {time.time() - t0:5.1f}s")

with torch.no_grad():
    test_acc = (model(X_test.to(device)).argmax(dim=1) == y_test.to(device)).float().mean().item()
print(f"test accuracy {test_acc:.4f}")

Same network, same numbers, a third less code, and — more importantly — code whose structure survives
scaling up. When the model becomes a convolutional network with fifty layers, `model(x)` still reads the
same, and when the optimizer becomes Adam with a learning-rate schedule and weight decay, the training
loop still reads the same. Writing your own module is equally simple: subclass `nn.Module`, create
sub-modules in `__init__`, and compose them in `forward`. Autograd takes care of the rest, exactly as it
did for our raw tensors, because a module's parameters are nothing more than tensors with
`requires_grad=True` that the module keeps track of.

In [ ]:
class TwoLayerMLP(nn.Module):
    """A custom module: sub-modules are created in __init__ and composed in forward."""

    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


custom = TwoLayerMLP(D_IN, 64, 10).to(device)
for name, p in custom.named_parameters():
    print(f"{name:12s} shape {str(tuple(p.shape)):12s} requires_grad={p.requires_grad} is_leaf={p.is_leaf}")
out = custom(X_train[:8].to(device))
print("output shape:", tuple(out.shape), "| grad_fn:", out.grad_fn)
print("optimizers with the same interface:", [cls.__name__ for cls in (torch.optim.SGD, torch.optim.Adam, torch.optim.AdamW)])

Notebook 04 studies
what goes inside `optimizer.step()` (momentum, Adam, schedules) and what goes between the layers
(initialization, normalization, regularization); Notebook 05 · The
PyTorch Workflow and Best Practices turns the loop above into a reusable, checkpointed, logged pipeline.
Everything they build rests on the two facts established here: a tensor can live on a GPU, and a tensor
can remember how it was computed.

> **Key idea.** `nn.Module` is a container for parameter tensors plus a `forward`; `torch.optim` is a loop
> over those tensors that applies an update rule. Neither adds anything that raw tensors and autograd
> cannot do — they only remove the bookkeeping.

## Exercises

### Exercise 1 (warm-up): `sigmoid` and `softmax` for the `Value` engine

Add a `sigmoid` method to `Value` with its own backward rule, $\sigma'(x) = \sigma(x)(1 - \sigma(x))$, and a
function `softmax(values)` that maps a list of `Value`s to a list of `Value`s using the existing `exp` and
division operators (subtract the maximum first for stability). Verify both against `torch.autograd` on a
small expression.

**Solution to Exercise 1**

The sigmoid follows the template of `tanh`: compute the output, store it in the closure, multiply the
upstream gradient by $s(1-s)$. Attaching the method to the class after the fact (`Value.sigmoid = ...`)
avoids re-defining the whole class. The softmax needs no backward of its own — it is composed from
operations the engine already differentiates, which is exactly how most functions in PyTorch are built.
Subtracting the maximum (a plain float, so it contributes no gradient) leaves the result unchanged
mathematically.

In [ ]:
def _value_sigmoid(self):
    s = 1.0 / (1.0 + math.exp(-self.data))
    out = Value(s, (self,), "sigmoid")

    def _backward():
        self.grad += s * (1.0 - s) * out.grad
    out._backward = _backward
    return out


Value.sigmoid = _value_sigmoid


def value_softmax(values):
    """Softmax over a list of Values, composed from exp and division."""
    m = max(v.data for v in values)                       # a constant: shifts logits, not gradients
    exps = [(v - m).exp() for v in values]
    total = sum(exps, Value(0.0))
    return [e / total for e in exps]


z_vals = [Value(0.5), Value(-1.0), Value(2.0)]
probs = value_softmax(z_vals)
expr = probs[1] * 3.0 + z_vals[0].sigmoid() + probs[2].log()
expr.backward()

z_t = torch.tensor([0.5, -1.0, 2.0], dtype=torch.float64, requires_grad=True)
p_t = torch.softmax(z_t, dim=0)
(p_t[1] * 3.0 + torch.sigmoid(z_t[0]) + torch.log(p_t[2])).backward()
print("softmax values engine:", [round(p.data, 6) for p in probs], "| torch:", [round(v, 6) for v in p_t.tolist()])
print("gradients engine     :", [round(v.grad, 6) for v in z_vals], "| torch:", [round(v, 6) for v in z_t.grad.tolist()])

The values and gradients match `torch.autograd`. Notice that `softmax` did not need a backward rule of its
own: composition is the whole point of an autograd engine.

### Exercise 2: a `Value`-based cross-entropy

Implement `value_cross_entropy(logits, target)` for a list of `Value` logits and an integer class index,
using the log-sum-exp form $\ell = \log \sum_i e^{z_i} - z_{y}$ rather than `log(softmax(...)[y])`. Check that
the gradient with respect to the logits equals $\mathbf{p} - \mathbf{y}$ (Notebook 02 · Neural Networks from
Scratch (NumPy), Section 5) and that it
matches `F.cross_entropy`.

**Solution to Exercise 2**

The log-sum-exp is stabilised by subtracting the maximum logit $m$ (a constant) and adding it back:
$\log \sum_i e^{z_i} = m + \log \sum_i e^{z_i - m}$. The gradient check confirms the $\mathbf{p} - \mathbf{y}$ result
for a third time, now produced by an engine that knows nothing about softmax.

In [ ]:
def value_cross_entropy(logits, target):
    """-log softmax(logits)[target], computed via a stable log-sum-exp."""
    m = max(v.data for v in logits)
    lse = sum(((v - m).exp() for v in logits), Value(0.0)).log() + m
    return lse - logits[target]


z_vals = [Value(1.0), Value(2.0), Value(-0.5), Value(0.3)]
loss_v = value_cross_entropy(z_vals, target=1)
loss_v.backward()

z_t = torch.tensor([1.0, 2.0, -0.5, 0.3], dtype=torch.float64, requires_grad=True)
loss_t = F.cross_entropy(z_t.unsqueeze(0), torch.tensor([1]))
loss_t.backward()
p_minus_y = torch.softmax(z_t.detach(), dim=0); p_minus_y[1] -= 1.0

print(f"loss engine {loss_v.data:.6f} | torch {loss_t.item():.6f}")
print("grad engine :", [round(v.grad, 6) for v in z_vals])
print("grad torch  :", [round(v, 6) for v in z_t.grad.tolist()])
print("p - y       :", [round(v, 6) for v in p_minus_y.tolist()])

### Exercise 3: the Hessian of a quadratic with autograd

For $f(\mathbf{x}) = \tfrac12 \mathbf{x}^\top A \mathbf{x} + \mathbf{b}^\top \mathbf{x}$ with a symmetric $A$, compute the
Hessian in two ways: (a) with nested `torch.autograd.grad` calls (differentiate the gradient one
component at a time, using `create_graph=True`), and (b) with `torch.func.hessian`. Confirm that both
equal $A$ and that the gradient at $\mathbf{x}_0$ equals $A\mathbf{x}_0 + \mathbf{b}$.

**Solution to Exercise 3**

The nested-`grad` route is the general recipe for any second-order quantity: compute the gradient with
`create_graph=True`, then differentiate each of its components. It costs one backward pass per row of the
Hessian, which is why `torch.func.hessian` (or Hessian-vector products via a single extra backward pass)
is preferred for large problems.

In [ ]:
torch.manual_seed(SEED)
M = torch.randn(4, 4, dtype=torch.float64)
A = M + M.T                                            # symmetric
b_vec = torch.randn(4, dtype=torch.float64)
quad_lin = lambda x: 0.5 * x @ A @ x + b_vec @ x

x0 = torch.randn(4, dtype=torch.float64, requires_grad=True)
g = torch.autograd.grad(quad_lin(x0), x0, create_graph=True)[0]          # gradient, still differentiable
H_rows = [torch.autograd.grad(g[i], x0, retain_graph=True)[0] for i in range(4)]
H_nested = torch.stack(H_rows)
H_func = torch.func.hessian(quad_lin)(x0.detach())

print("gradient == A x0 + b:", torch.allclose(g.detach(), A @ x0.detach() + b_vec))
print("nested-grad Hessian == A:", torch.allclose(H_nested, A), "| torch.func.hessian == A:", torch.allclose(H_func, A))
print("Hessian is symmetric:", torch.allclose(H_nested, H_nested.T))

### Exercise 4: find the in-place bug

The following function computes a small loss from a weight matrix, but calling `backward()` on its output
fails. Explain why, and fix it without changing what the function computes.

```python
def buggy_loss(x, w):
    z = x @ w
    h = torch.tanh(z)
    h[h < 0] = 0.0            # keep only the positive activations
    return (h ** 2).sum()
```

**Solution to Exercise 4**

`torch.tanh` saves its *output* `h` for the backward pass (its derivative is $1 - h^2$). The masked
assignment `h[h < 0] = 0.0` modifies `h` in place, bumping its version counter, so when autograd later
reaches the `tanh` node it finds that the saved tensor has changed and refuses to continue. The fix is
any out-of-place equivalent: `torch.where(h < 0, torch.zeros_like(h), h)`, `h.clamp(min=0)`, or
`torch.relu(h)`. The cell shows the error message and confirms that the fixed version produces a gradient.

In [ ]:
def buggy_loss(x, w):
    z = x @ w
    h = torch.tanh(z)
    h[h < 0] = 0.0
    return (h ** 2).sum()


def fixed_loss(x, w):
    z = x @ w
    h = torch.tanh(z)
    h = torch.where(h < 0, torch.zeros_like(h), h)      # out of place: h is a new tensor
    return (h ** 2).sum()


torch.manual_seed(SEED)
x_in = torch.randn(5, 3)
w_bug = torch.randn(3, 4, requires_grad=True)
try:
    buggy_loss(x_in, w_bug).backward()
except RuntimeError as err:
    print("buggy_loss ->", str(err)[:110], "...")

fixed_loss(x_in, w_bug).backward()
print("fixed_loss -> gradient shape", tuple(w_bug.grad.shape), "| finite:", torch.isfinite(w_bug.grad).all().item())

### Exercise 5 (challenge): a custom `Function` for the swish activation

The swish (or SiLU) activation is $\operatorname{swish}(x) = x\,\sigma(x)$. Implement it as a
`torch.autograd.Function` with a hand-derived backward, verify it with `torch.autograd.gradcheck`, and
compare its gradient with the one autograd derives automatically for the expression `x * torch.sigmoid(x)`.

**Solution to Exercise 5**

Differentiating with the product rule and $\sigma' = \sigma(1 - \sigma)$:

$$
\frac{d}{dx}\big[x\,\sigma(x)\big] = \sigma(x) + x\,\sigma(x)\big(1 - \sigma(x)\big) = \sigma(x)\big(1 + x\,(1 - \sigma(x))\big).
$$

We save the input in `forward` and evaluate the expression in `backward`. `gradcheck` compares this
rule against finite differences in float64; the comparison with the gradient autograd derives on its own for
`x * torch.sigmoid(x)` shows that the manual and automatic gradients coincide, which is the usual outcome — the custom `Function` only pays off when it is
faster or more stable than the composed version, as in Section 4.9.

In [ ]:
class Swish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * torch.sigmoid(x)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        s = torch.sigmoid(x)
        return grad_output * s * (1.0 + x * (1.0 - s))


swish = Swish.apply
x_sw = torch.randn(8, dtype=torch.float64, requires_grad=True)
print("gradcheck passes:", torch.autograd.gradcheck(swish, (x_sw,), eps=1e-6, atol=1e-5))

grad_custom = torch.autograd.grad(swish(x_sw).sum(), x_sw)[0]
grad_auto = torch.autograd.grad((x_sw * torch.sigmoid(x_sw)).sum(), x_sw)[0]
print("custom backward == autograd of x * sigmoid(x):", torch.allclose(grad_custom, grad_auto))
print("matches F.silu:", torch.allclose(swish(x_sw), F.silu(x_sw)))

## Summary

- Tensors are NumPy-like arrays with a device, a float32 default and an optional computation history; shapes, indexing and broadcasting work as in NumPy, and `dim`/`keepdim`, `einsum`, `stack`/`cat` cover the reshaping needs of every later notebook.
- Views (`view`, `reshape`, `transpose`, slicing) share memory with their source; in-place operations (underscore methods, index assignment) modify it, which autograd tracks with a version counter.
- Automatic differentiation records a computational graph during the forward pass and applies the chain rule to every elementary operation; reverse mode computes the gradient of one scalar with respect to all inputs in one backward sweep, which is why it is the right tool for training.
- A scalar autograd engine needs only: a `data`/`grad` pair per node, a local backward closure per operation that *adds* its contribution to the inputs' gradients, and a reverse topological traversal — under a hundred lines of Python.
- The engine works but is slow because it creates one node per scalar operation; tensors record one node per matrix operation and run the arithmetic in optimised kernels.
- In `torch.autograd`: `requires_grad=True` on leaves, `loss.backward()` fills `.grad`, gradients accumulate (zero them), the graph is freed after `backward()` (use `retain_graph=True` if you need it again), evaluation and parameter updates go under `torch.no_grad()`, and `detach()` cuts a tensor out of the graph.
- `backward(v)` computes the vector-Jacobian product $\mathbf{v}^\top J$; `torch.autograd.grad(..., create_graph=True)` enables higher-order derivatives; `torch.func.jacrev`/`hessian`/`vmap` give whole Jacobians, Hessians and per-example gradients.
- A custom `torch.autograd.Function` pairs a `forward` with a hand-written `backward`; always verify it with `gradcheck` in float64.
- With raw tensors and autograd, the MNIST MLP of Notebook 02 · Neural Networks from Scratch (NumPy) shrinks from about 150 lines to about 40 with identical accuracy; `nn.Module` and `torch.optim` remove the remaining bookkeeping without adding new capabilities.

## Further reading

- PyTorch documentation, *Autograd mechanics*: https://docs.pytorch.org/docs/stable/notes/autograd.html — the authoritative description of leaves, in-place checks and the version counter.
- PyTorch documentation, *torch.autograd*: https://docs.pytorch.org/docs/stable/autograd.html and *torch.func*: https://docs.pytorch.org/docs/stable/func.html
- PyTorch documentation, *Tensor views*: https://docs.pytorch.org/docs/stable/tensor_view.html and *Broadcasting semantics*: https://docs.pytorch.org/docs/stable/notes/broadcasting.html
- Karpathy, A., 2020. *micrograd*: https://github.com/karpathy/micrograd, and the accompanying lecture "The spelled-out intro to neural networks and backpropagation: building micrograd" (2022). The inspiration for Section 3.
- Baydin, A. G., Pearlmutter, B. A., Radul, A. A. and Siskind, J. M., 2018. *Automatic differentiation in machine learning: a survey*. Journal of Machine Learning Research 18(153). A thorough survey of forward and reverse mode.
- Griewank, A. and Walther, A., 2008. *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation*, 2nd edition. SIAM. The standard reference on AD.
- Paszke, A. et al., 2019. *PyTorch: An Imperative Style, High-Performance Deep Learning Library*. NeurIPS. Describes the design decisions behind eager execution and autograd.
- Goodfellow, I., Bengio, Y. and Courville, A., 2016. *Deep Learning*. MIT Press, Section 6.5 (Back-Propagation and Other Differentiation Algorithms). https://www.deeplearningbook.org/
- Zhang, A., Lipton, Z. C., Li, M. and Smola, A. J. *Dive into Deep Learning*, Chapter 2 (Preliminaries: data manipulation, automatic differentiation). https://d2l.ai/